# Meal history and postprandial glucose response outcomes

This notebook derives dietary-history features, premeal glucose summaries and two-hour postprandial glucose response (PPGR) outcomes from the merged meal table. It also calculates meal intervals and identifies standardized challenges. The final analysis eligibility filters are applied later in `Code/utils.py`.

## Notebook flow

1. Load participant metadata, CGM measurements and the merged meals from preprocessing notebook 01.
2. Align meal-history calculations to nearby premeal glucose minima and summarize prior food intake and glycemic state.
3. Calculate two-hour PPGR outcomes and glucose traces with interpolated window boundaries and a maximum permitted gap between readings.
4. Remove meals without a PPGR trace, fill missing nutrient totals, and calculate meal intervals and clock-time features.
5. Identify standardized glucose-drink, white-bread and white-bread-with-butter challenges using food identifiers and carbohydrate amounts, then export the prepared table.

## Inputs

Source files in `Data/raw data/`: `metadata.csv`, `cgm_data.csv` and `meal_data.csv`. The merged meal table is produced by preprocessing notebook 01. The notebook also requires the `tqdm_joblib` package for parallel progress reporting.

## Outputs

| File | Contents |
| --- | --- |
| `Data/meal_ppgr.csv` or `Data/meal_ppgr.zip` | Prepared meal records with nutrient and glucose-history features, PPGR outcomes and response arrays, meal intervals, timing features and standardized-meal labels. |

The export uses `MEAL_DATA_PATH` from `Code/data_paths.py`. It writes the ZIP when that archive exists and the uncompressed CSV is absent; otherwise it writes the CSV. Dataset previews and processing counts are displayed in the notebook.

## 1. Setup

Resolve project paths and import the preprocessing packages.

In [1]:
from pathlib import Path
import sys

# Find the project when launched from its root, Code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "Code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
from data_paths import (
    RAW_DATA_DIR, RAW_METADATA_PATH, RAW_CGM_PATH, RAW_FOOD_PATH,
    MERGED_MEALS_PATH, CLEANED_FOOD_PATH,
)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, timedelta

from scipy.signal import argrelmin

pd.set_option('mode.chained_assignment', None)
import warnings
warnings.filterwarnings(
    'ignore',
    message='A value is trying to be set on a copy of a DataFrame',
)
warnings.filterwarnings(
    'ignore',
    message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated',
)
warnings.filterwarnings("ignore")

## 2. Load participant, CGM and merged-meal data

Align participant keys, prepare glucose measurements and load the output of preprocessing notebook 01.

In [2]:
meta_data = pd.read_csv(RAW_METADATA_PATH).reset_index(drop=True)

meta_data.rename(columns={'subject_app_key':'subject_key'}, inplace=True)

if 'id' in meta_data.columns:
    id_to_subjectKey = dict(zip(meta_data['id'], meta_data['subject_key']))
    meta_data.rename(columns={'id':'fay-id'}, inplace=True)
else:
    id_to_subjectKey = dict(zip(meta_data['fay-id'], meta_data['subject_key']))

In [3]:
df_gluc = pd.read_csv(RAW_CGM_PATH, index_col=0).reset_index(drop=False)


df_gluc.rename(columns={'read_at':'time', 'user_id':'id', 'val':'gl_mmol'}, inplace=True)

df_gluc['subject_key'] = df_gluc['id'].map(id_to_subjectKey)
df_gluc = df_gluc[~df_gluc['subject_key'].isna()]

df_gluc['time'] = pd.to_datetime(df_gluc['time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
# Remove rows with NaT in the 'time' column
df_gluc = df_gluc.dropna(subset=['time'])

df_gluc.sort_values(by=['subject_key', 'time'], inplace=True)
df_gluc["mg_dL"] = df_gluc["gl_mmol"] * 18
print(df_gluc.shape)
df_gluc.head()

(1522406, 5)


,time,id,gl_mmol,subject_key,mg_dL
0,2018-11-26 08:00:00,5,6.06,02ae3856ca04,109.08
1,2018-11-26 08:15:00,5,7.55,02ae3856ca04,135.90
2,2018-11-26 08:30:00,5,8.51,02ae3856ca04,153.18
3,2018-11-26 08:45:00,5,7.39,02ae3856ca04,133.02
4,2018-11-26 09:00:00,5,6.80,02ae3856ca04,122.40


In [4]:
merged_foods_users = pd.read_csv(MERGED_MEALS_PATH)
merged_foods_users["eaten_at"] = pd.to_datetime(merged_foods_users["eaten_at"])
merged_foods_users.shape

(105769, 47)

## 3. Define glucose alignment and response calculations

Define premeal alignment, interpolation and PPGR calculations. The later interpolation-enabled definition is used by the outcome-processing step.

In [5]:
def find_last_local_min_row(df, gluc_var='mg_dL'):
    # Ensure the DataFrame is sorted by the time column
    df = df.sort_values(by='time')
    # Initialize variables to store the last local minimum index
    last_local_min_index = None
    # Initialize variables to track the global minimum
    global_min = df[gluc_var].iloc[0]
    global_min_index = 0
    # Iterate through the DataFrame in reverse order
    for i in range(len(df) - 2, -1, -1):
        if df[gluc_var].iloc[i] < df[gluc_var].iloc[i - 1] and df[gluc_var].iloc[i] < df[gluc_var].iloc[i + 1]:
            last_local_min_index = i
            break
    # Iterate through the entire DataFrame to find the global minimum
    for i in range(len(df)-1):
        if df[gluc_var].iloc[i] < global_min:
            global_min = df[gluc_var].iloc[i]
            global_min_index = i
    # Return the row corresponding to the last local minimum index
    if last_local_min_index is None : 
        last_local_min_row = df.iloc[global_min_index]
    else : 
        last_local_min_row = df.iloc[last_local_min_index]
    return last_local_min_row

def closest_gluc_timepoint(x, my_glu):
    closest_entry = my_glu.iloc[(my_glu['time'] - x).abs().argsort()[:1]]
    return closest_entry['time'].values[0]

def retrieve_gluc_vals(x, my_glu, gluc_var="mg_dL"):
    gluc_vals = my_glu[my_glu["time"] == x][gluc_var]
    if len(gluc_vals) == 0:
        gluc_vals = my_glu.loc[my_glu["time"] == x, gluc_var]
        if len(gluc_vals) == 0:
            return None
        else : 
            return gluc_vals.values.mean()
    return gluc_vals.values.mean()

In [6]:
def shift_times_to_argrelmin(
    food_df,
    gluc_df,
    timewindow=30,            # short look-back (min), used by default
    long_window=30,          # extended look-back (min) after a long fast
    fasting_threshold=120,    # min since previous meal that counts as "fasted"
    eating_time_col="eaten_at",
    order=1,                  # default neighbourhood for normal window
    long_order=1,             # increased order for long window
    gluc_var='mg_dL',
    keep_tol_min=15,          # if eaten_at is within this of the minima, KEEP eaten_at
):
    shifted_times = []
    food_df = food_df.sort_values(by=eating_time_col).copy()
    gluc_df = gluc_df.sort_values(by="time").copy()
    meal_times = pd.to_datetime(food_df[eating_time_col]).reset_index(drop=True)
    for i, mealtime in enumerate(meal_times):
        # 1. time since the previous logged meal (this user)
        gap_min = np.inf if i == 0 else (mealtime - meal_times.iloc[i - 1]).total_seconds() / 60
        # 2. choose the window and order
        if gap_min >= fasting_threshold:
            window = long_window
            current_order = long_order
        else:
            window = timewindow
            current_order = order

        window = min(window, gap_min)
        start_time = mealtime - timedelta(minutes=window)
        local_gluc = gluc_df[(gluc_df["time"] >= start_time) &
                             (gluc_df["time"] <= mealtime)].reset_index(drop=True)
        if local_gluc.empty:
            shifted_times.append(np.nan)
            continue
        # 3. last local minimum before the meal = onset of the rise
        minima_indices = argrelmin(local_gluc[gluc_var].values, order=current_order)[0]
        if minima_indices.shape[0] == 0:
            chosen = local_gluc.iloc[np.argmin(local_gluc[gluc_var].values)]  # fallback: global min in window
        else:
            chosen = local_gluc.iloc[minima_indices[-1]]

        chosen_time = chosen["time"]
        # if eaten_at is already within keep_tol_min of the detected onset, keep
        # the real eaten_at instead of snapping back to the minima (reduces bias /
        # avoids forcing everything onto the CGM grid). interpolation handles t=0.
        if abs((mealtime - chosen_time).total_seconds()) / 60.0 <= keep_tol_min:
            shifted_times.append(mealtime)
        else:
            shifted_times.append(chosen_time)

    food_df['shifted_eaten_at'] = shifted_times
    return food_df

In [7]:
def calculate_PPGR_iAUC(
    df,
    key,
    food_intake_time,
    post_meal_duration=120,
    verbose=False
):
    """
    Calculate PPGR incremental AUC (iAUC) for a specific subject.

    Parameters
    ----------
    df : pd.DataFrame
        Glucose readings containing subject_key, time and mg_dL.
    key :
        Subject identifier.
    food_intake_time :
        Time when food was consumed.
    post_meal_duration : int, default=120
        Duration in minutes after food intake.
    verbose : bool, default=False
        Print diagnostic messages.

    Returns
    -------
    tuple
        (iAUC, baseline), or (None, None) if calculation isn't possible.
    """

    # Safely convert food timestamp
    food_intake_time = pd.to_datetime(
        food_intake_time,
        errors="coerce"
    )

    if pd.isna(food_intake_time):
        if verbose:
            print(f"{key}: invalid food intake time.")
        return None, None

    # Extract subject glucose readings
    user_data = df[df["subject_key"] == key].copy()

    if user_data.empty:
        if verbose:
            print(f"{key}: no glucose data found.")
        return None, None

    # Ensure valid datetime values
    user_data["time"] = pd.to_datetime(
        user_data["time"],
        errors="coerce"
    )

    # Remove invalid glucose timestamps
    user_data = user_data.dropna(
        subset=["time", "mg_dL"]
    )

    if user_data.empty:
        if verbose:
            print(f"{key}: no valid glucose readings found.")
        return None, None

    # Preserve chronological order for first/last-row indexing.
    user_data = user_data.sort_values("time")

    # Find closest glucose reading to meal time
    time_diff = (user_data["time"] - food_intake_time).abs()

    closest_idx = time_diff.idxmin()
    closest_time = user_data.loc[closest_idx, "time"]

    # Require glucose reading within 16 minutes of meal
    closest_gap = abs(food_intake_time - closest_time)

    if closest_gap > pd.Timedelta(minutes=16):
        if verbose:
            print(
                f"{food_intake_time}: {key} "
                f"closest reading is {closest_gap.total_seconds()/60:.1f} "
                f"minutes away ({closest_time})."
            )
        return None, None

    # End of post-meal window
    end_time = closest_time + pd.Timedelta(
        minutes=post_meal_duration
    )

    # Extract post-meal readings
    post_meal_data = user_data[
        (user_data["time"] >= closest_time)
        & (user_data["time"] <= end_time)
    ].copy()

    post_meal_data = post_meal_data.sort_values("time")

    # Require minimum number of readings
    minimum_readings = int(post_meal_duration / 30)

    if len(post_meal_data) < minimum_readings:
        if verbose:
            print(
                f"{food_intake_time}: {key} "
                f"only {len(post_meal_data)} readings found "
                f"between {closest_time} and {end_time}."
            )
        return None, None

    # Check gap at beginning
    start_gap = (
        post_meal_data["time"].iloc[0] - closest_time
    ).total_seconds() / 60

    if start_gap > 30:
        if verbose:
            print(
                f"{food_intake_time}: {key} "
                f"start gap is {start_gap:.1f} minutes."
            )
        return None, None

    # Check gap at end
    end_gap = (
        end_time - post_meal_data["time"].iloc[-1]
    ).total_seconds() / 60

    if end_gap > 30:
        if verbose:
            print(
                f"{food_intake_time}: {key} "
                f"end gap is {end_gap:.1f} minutes."
            )
        return None, None

    # Baseline
    baseline = post_meal_data["mg_dL"].iloc[0]

    if pd.isna(baseline):
        return None, None

    # Increment above baseline
    post_meal_data["adjusted_val"] = (
        post_meal_data["mg_dL"] - baseline
    ).clip(lower=0)

    # Minutes since closest meal-time reading
    time_intervals = (
        post_meal_data["time"] - closest_time
    ).dt.total_seconds() / 60

    # Incremental area under curve
    iAUC = np.trapz(
        post_meal_data["adjusted_val"],
        x=time_intervals
    )

    return round(iAUC, 2), baseline

In [8]:
def calculate_time_above_baseline(times, values, baseline):
    """
    Compute total duration (minutes) that glucose is above baseline.

    Parameters
    ----------
    times : pd.Series or array-like of datetime
    values : pd.Series or array-like of glucose values
    baseline : float

    Returns
    -------
    duration_minutes : float
    """

    duration = 0.0

    for i in range(len(values) - 1):
        t0 = pd.to_datetime(times.iloc[i])
        t1 = pd.to_datetime(times.iloc[i + 1])

        y0 = values.iloc[i] - baseline
        y1 = values.iloc[i + 1] - baseline

        dt = (t1 - t0).total_seconds() / 60

        # Entire segment above baseline
        if y0 > 0 and y1 > 0:
            duration += dt

        # Crossing upward
        elif y0 <= 0 and y1 > 0:
            frac = -y0 / (y1 - y0)
            crossing_time = frac * dt
            duration += dt - crossing_time

        # Crossing downward
        elif y0 > 0 and y1 <= 0:
            frac = y0 / (y0 - y1)
            crossing_time = frac * dt
            duration += crossing_time

        # Both below baseline -> contribute nothing

    return duration

def _split_at_baseline(t, v):
    T, V = [t[0]], [v[0]]
    for i in range(len(v) - 1):
        if (v[i] > 0) != (v[i+1] > 0) and v[i] != v[i+1]:
            frac = -v[i] / (v[i+1] - v[i])
            T.append(t[i] + frac * (t[i+1] - t[i])); V.append(0.0)
        T.append(t[i+1]); V.append(v[i+1])
    return np.asarray(T), np.asarray(V)

In [9]:
from scipy.stats import linregress

def calculate_PPGR_iAUC_minima(df, key, food_intake_time, gluc_threshold, 
                               post_meal_duration=120, verbose=False, timeshift=0, minima=False, df_minima= None, gluc_var='mg_dL',
                              interpolation=False, freq="15min",):
    """
    Calculate the iAUC for PPGR for a specific user based on the time of food intake.
    
    Parameters:
    - df: DataFrame containing the glucose readings
    - key: ID of the user for whom iAUC should be calculated
    - food_intake_time: Time when the food was consumed
    - post_meal_duration: Duration in minutes after food intake for the iAUC calculation (default is 120 minutes)
    
    Returns:
    - iAUC value for PPGR
    """
    # Extract user's glucose readings
    user_data = df[df['subject_key'] == key].copy().sort_values(by="time")
    # Find the closest reading time to the food intake time
    closest_time = user_data.iloc[(user_data["time"] - pd.to_datetime(food_intake_time)).abs().argsort()[:1]]["time"].values[0]    
    #closest_time = pd.to_datetime(food_intake_time)
    if abs(food_intake_time - closest_time ) > pd.Timedelta(minutes=16):
        if verbose:
            print(f"{str(food_intake_time)}: {key} The closest reading time is 30min away; {closest_time}.")
        return None, None
    if minima :  # Start the PPGR window at the glucose minimum.
        # dataframe containing the the possible PPGR window begginings
        possible_begginings = user_data[(user_data['time'] > closest_time - pd.Timedelta(minutes=60)) & (user_data['time'] <= closest_time + pd.Timedelta(minutes=0))].copy()
        last_local_minima_row = find_last_local_min_row(possible_begginings)
        closest_time = last_local_minima_row["time"]
        right_values = possible_begginings[possible_begginings["time"] >= last_local_minima_row["time"]].reset_index(drop=True)

        for i in range(1, len(right_values[gluc_var])):
            if (right_values[gluc_var][i] - right_values[gluc_var][i - 1]) < gluc_threshold :
                closest_time = right_values["time"][i]
            else : 
                break
    # Define the end time based on the specified post meal duration
    end_time = pd.to_datetime(closest_time) + pd.Timedelta(minutes=post_meal_duration)
    # Filter the readings for the specified duration after food intake
    post_meal_data = user_data[(user_data['time'] >= closest_time) & (user_data['time'] <= end_time)].copy()
    if len(post_meal_data) < int(post_meal_duration/30):
        if verbose:
            print(f"{str(food_intake_time)}: {key} Very few readings were found between {closest_time} and {end_time}.")
        #return None, None
    # Check the gap between the start time and the first reading
    start_gap = (pd.to_datetime(post_meal_data['time'].iloc[0]) - pd.to_datetime(closest_time)).seconds / 60
    if start_gap > 30 and verbose:
        print(f"{str(food_intake_time)}: {key} The gap between the food intake time and the first glucose reading is {start_gap} minutes.")
        #return None, None
    
    # Check the gap between the end time and the last reading
    end_gap = (end_time - pd.to_datetime(post_meal_data['time'].iloc[-1])).seconds / 60
    if end_gap > 30 and verbose:
        print(f"{str(food_intake_time)}: {key} The gap between the end time and the last glucose reading is {end_gap} minutes.")
        #return None, None
    # Subtract the baseline value from all readings
    baseline = post_meal_data[gluc_var].iloc[0]  # Assuming the first value is the baseline
    post_meal_data.loc[:, 'adjusted_val'] = post_meal_data[gluc_var] - baseline
    # Convert 'time' to a series of time intervals in minutes from the closest time
    time_intervals = (post_meal_data['time'] - pd.to_datetime(closest_time)).dt.total_seconds() / 60
    # Calculate the area using the trapezoidal rule for values above the baseline


    ######## PPGR metrics
    positive_iAUC = round(np.trapz(post_meal_data['adjusted_val'].clip(lower=0), x=time_intervals),3)
    negative_iAUC = round(np.trapz(post_meal_data['adjusted_val'].clip(upper=0), x=time_intervals), 3)
    ppgr_array = post_meal_data[gluc_var].to_list()
    net_iAUC = round(np.trapz(post_meal_data["adjusted_val"], x=time_intervals), 3)

    # delta glucose
    max_idx = post_meal_data[gluc_var].idxmax()
    max_glucose = post_meal_data.loc[max_idx, gluc_var]
    delta_glucose =  max_glucose -baseline
    # time to peak
    time_of_max = post_meal_data.loc[max_idx, "time"]
    time_to_max = (
        pd.to_datetime(time_of_max) - pd.to_datetime(closest_time)
    ).total_seconds() / 60

    # time above baseline
    peak_duration = calculate_time_above_baseline(
                                    post_meal_data["time"],
                                    post_meal_data[gluc_var],
                                    baseline
                                )

    # recovery fraction: how much of the peak excursion is still present at window end
    # R ~ 0 -> fully recovered to baseline; R ~ 1 -> still near peak (monotonic responses)
    DELTA_FLOOR = 1.0  # mg/dL; below this, response is too flat for R to be meaningful

    # value at the fixed window end (t = post_meal_duration), interpolated onto the grid.
    # adjusted_val is already (glucose - baseline), so this is (G_end - baseline) directly.
    residual_end = float(np.interp(post_meal_duration, time_intervals, post_meal_data['adjusted_val']))

    if delta_glucose >= DELTA_FLOOR:
        recovery_percentage = round(1 - (residual_end / delta_glucose), 3)
    else:
        recovery_percentage = 0  # non-response: R undefined, flag rather than divide


    # 0->60mn burden and 60->120mn burden: 
    # split the net iAUC into 0–60 and 60–120 min windows
    split = 60
    # interpolate the value at the 60-min boundary so it's shared by both windows
    val_at_split = float(np.interp(split, time_intervals, post_meal_data['adjusted_val']))
    
    t = time_intervals.to_numpy()
    v = post_meal_data['adjusted_val'].to_numpy()
    
    # include the exact 60-min point as a boundary node in each window
    t0, v0 = np.append(t[t < split], split),       np.append(v[t < split], val_at_split)
    t1, v1 = np.insert(t[t > split], 0, split),     np.insert(v[t > split], 0, val_at_split)
    
    net_iAUC_0_60   = round(np.trapz(v0, x=t0), 3)
    net_iAUC_60_120 = round(np.trapz(v1, x=t1), 3)

    # nadir: absolute minimum glucose post-peak
    # nadir_delta: nadir minus baseline (positive = stayed above baseline,
    #              negative = dropped below baseline / reactive dip)
    # literature standard: nadir is absolute, delta is relative to baseline
    
    post_peak_data = post_meal_data[post_meal_data['time'] > time_of_max].copy()
    
    if len(post_peak_data) == 0:
        # peak at last measurement — no post-peak window
        nadir_glucose = max_glucose          # best we can do
        nadir_delta   = round(max_glucose - baseline, 3)  # still above baseline
    
    elif delta_glucose < DELTA_FLOOR:
        # no meaningful excursion — nadir is meaningless as a reactive dip metric
        # report the absolute nadir but flag delta as 0
        nadir_glucose = round(post_peak_data[gluc_var].min(), 3)
        nadir_delta   = 0.0
    
    else:
        # normal case: real excursion occurred
        nadir_glucose = round(post_peak_data[gluc_var].min(), 3)
        nadir_delta   = round(nadir_glucose - baseline, 3)
        # nadir_delta > 0 : recovered but stayed above baseline (no reactive dip)
        # nadir_delta < 0 : dropped below baseline (reactive hypoglycemia)
        # nadir_delta = 0 : recovered exactly to baseline

    # ascending slope
    ASCENT_FLOOR = 1.0  # Minimum excursion for the ratio, mg/dL.

    if delta_glucose < ASCENT_FLOOR:          # subsumes time_to_max == 0
        ascending_slope = 0.0
    else:
        ascending_slope = round(delta_glucose / time_to_max, 4)

    # recovery slope: rate of glucose descent after peak (mg/dL per minute)
    # computed over the post-peak portion of the window
    # negative = recovering
    
    DELTA_FLOOR_SLOPE = 1.0  # same guard as recovery_percentage
    
    desc = post_meal_data[post_meal_data['time'] >= time_of_max]
    tt   = (desc['time'] - pd.to_datetime(closest_time)).dt.total_seconds().to_numpy() / 60
    gg   = desc[gluc_var].to_numpy()
    
    if len(gg) < 2 or delta_glucose < ASCENT_FLOOR:
        recovery_slope = 0.0
    else:
        seg = np.diff(gg) / np.diff(tt)
        recovery_slope = round(min(float(np.min(seg)), 0.0), 4)   # most negative = fastest fall

    end_delta = round(residual_end, 3)

        
    return positive_iAUC , baseline, pd.to_datetime(closest_time), ppgr_array, net_iAUC, negative_iAUC, time_to_max, delta_glucose, peak_duration, recovery_percentage, net_iAUC_0_60, net_iAUC_60_120, recovery_slope, nadir_delta,end_delta, max_glucose 



In [10]:
import numpy as np
import pandas as pd
from scipy.stats import linregress


def interpolate_glucose_at(data, target_time, gluc_var="mg_dL",
                           time_col="time", tol_min=10):
    """
    Estimate glucose at an exact target_time by linear interpolation between the
    nearest reading strictly before and strictly after it.

    `data` must be sorted by time. Returns the interpolated value, or None if
    there is no reading within tol_min on EITHER side (i.e. a dropout at the edge).
    """
    target_time = pd.to_datetime(target_time)
    times = data[time_col]

    exact = data.loc[times == target_time, gluc_var]
    if len(exact):
        return float(exact.iloc[0])

    before = data[times < target_time]
    after  = data[times > target_time]
    if before.empty or after.empty:
        return None                      # target outside CGM coverage

    b, a = before.iloc[-1], after.iloc[0]
    dt_before = (target_time - b[time_col]).total_seconds() / 60.0
    dt_after  = (a[time_col] - target_time).total_seconds() / 60.0
    if dt_before > tol_min or dt_after > tol_min:
        return None                      # one side too far -> don't trust it

    frac = (target_time - b[time_col]) / (a[time_col] - b[time_col])
    return float(b[gluc_var] + frac * (a[gluc_var] - b[gluc_var]))


def calculate_PPGR_iAUC_minima(df, key, food_intake_time, gluc_threshold,
                               post_meal_duration=120, verbose=False, timeshift=0,
                               minima=False, df_minima=None, gluc_var='mg_dL',
                               interpolation=True, freq="15min", interp_tol_min=15, max_gap_min=None):
    """
    Calculate the iAUC for PPGR for a specific user based on the time of food intake.

    Parameters:
    - df: DataFrame containing the glucose readings
    - key: ID of the user for whom iAUC should be calculated
    - food_intake_time: Time when the food was consumed
    - post_meal_duration: Duration in minutes after food intake for the iAUC calculation (default is 120 minutes)
    - interpolation: if True, the window start (t=0) and end (t=post_meal_duration)
      glucose values are linearly interpolated between the bracketing readings,
      provided a reading exists within interp_tol_min on each side.
    - max_gap_min: if set, skip the window when any two consecutive REAL readings
      bracketing/inside the window are more than this many minutes apart.

    Returns:
    - iAUC value for PPGR (and other metrics)
    """
    # Extract user's glucose readings
    user_data = df[df['subject_key'] == key].copy().sort_values(by="time")
    food_intake_time = pd.to_datetime(food_intake_time)

    if interpolation:
        # ---- interpolation path: meal time is t=0, no snapping to closest reading ----
        start_time = food_intake_time
        end_time   = start_time + pd.Timedelta(minutes=post_meal_duration)

        # ---- gap check on REAL readings, before any interpolation ----
        if max_gap_min is not None:
            left   = user_data[user_data['time'] <= start_time].tail(1)
            inside = user_data[(user_data['time'] > start_time) &
                               (user_data['time'] < end_time)]
            right  = user_data[user_data['time'] >= end_time].head(1)
            covering = pd.concat([left, inside, right]).sort_values('time')

            if len(covering) < 2:
                if verbose:
                    print(f"{str(food_intake_time)}: {key} not enough readings "
                          f"bracketing the window; skipping.")
                return None, None

            largest_gap = float(
                covering['time'].diff().dt.total_seconds().div(60.0).max()
            )
            if largest_gap > max_gap_min:
                if verbose:
                    print(f"{str(food_intake_time)}: {key} largest gap in real "
                          f"readings is {largest_gap:.1f} min (> {max_gap_min}); "
                          f"skipping before interpolation.")
                return None, None

        start_val = interpolate_glucose_at(user_data, start_time,
                                           gluc_var=gluc_var, time_col="time",
                                           tol_min=interp_tol_min)
        end_val   = interpolate_glucose_at(user_data, end_time,
                                           gluc_var=gluc_var, time_col="time",
                                           tol_min=interp_tol_min)
        if start_val is None or end_val is None:
            if verbose:
                print(f"{str(food_intake_time)}: {key} no reading within "
                      f"{interp_tol_min} min of the window start and/or end; skipping.")
            return None, None

        # real readings strictly inside the window + the two interpolated edges
        interior = user_data[(user_data['time'] > start_time) &
                             (user_data['time'] < end_time)][['time', gluc_var]]
        edges = pd.DataFrame({'time':   [start_time, end_time],
                              gluc_var: [start_val,  end_val]})
        post_meal_data = (pd.concat([edges, interior], ignore_index=True)
                            .sort_values('time')
                            .reset_index(drop=True))
        closest_time = start_time          # t = 0 is the meal time itself

    else:
        # ---- original closest-reading path ----
        # Find the closest reading time to the food intake time
        closest_time = user_data.iloc[(user_data["time"] - food_intake_time).abs().argsort()[:1]]["time"].values[0]
        #closest_time = pd.to_datetime(food_intake_time)
        if abs(food_intake_time - closest_time) > pd.Timedelta(minutes=16):
            if verbose:
                print(f"{str(food_intake_time)}: {key} The closest reading time is 30min away; {closest_time}.")
            return None, None
        if minima:  # Start the PPGR window at the glucose minimum.
            # dataframe containing the the possible PPGR window begginings
            possible_begginings = user_data[(user_data['time'] > closest_time - pd.Timedelta(minutes=60)) & (user_data['time'] <= closest_time + pd.Timedelta(minutes=0))].copy()
            last_local_minima_row = find_last_local_min_row(possible_begginings)
            closest_time = last_local_minima_row["time"]
            right_values = possible_begginings[possible_begginings["time"] >= last_local_minima_row["time"]].reset_index(drop=True)

            for i in range(1, len(right_values[gluc_var])):
                if (right_values[gluc_var][i] - right_values[gluc_var][i - 1]) < gluc_threshold:
                    closest_time = right_values["time"][i]
                else:
                    break
        # Define the end time based on the specified post meal duration
        end_time = pd.to_datetime(closest_time) + pd.Timedelta(minutes=post_meal_duration)
        # Filter the readings for the specified duration after food intake
        post_meal_data = user_data[(user_data['time'] >= closest_time) & (user_data['time'] <= end_time)].copy()

        # ---- gap check on real readings (closest-reading path) ----
        if max_gap_min is not None and len(post_meal_data) >= 2:
            largest_gap = float(
                post_meal_data['time'].sort_values()
                .diff().dt.total_seconds().div(60.0).max()
            )
            if largest_gap > max_gap_min:
                if verbose:
                    print(f"{str(food_intake_time)}: {key} largest intra-window gap "
                          f"is {largest_gap:.1f} min (> {max_gap_min}); skipping.")
                return None, None

    if len(post_meal_data) < int(post_meal_duration/30):
        if verbose:
            print(f"{str(food_intake_time)}: {key} Very few readings were found between {closest_time} and {end_time}.")
        #return None, None
    # Check the gap between the start time and the first reading
    start_gap = (pd.to_datetime(post_meal_data['time'].iloc[0]) - pd.to_datetime(closest_time)).seconds / 60
    if start_gap > 30 and verbose:
        print(f"{str(food_intake_time)}: {key} The gap between the food intake time and the first glucose reading is {start_gap} minutes.")
        #return None, None

    # Check the gap between the end time and the last reading
    end_gap = (end_time - pd.to_datetime(post_meal_data['time'].iloc[-1])).seconds / 60
    if end_gap > 30 and verbose:
        print(f"{str(food_intake_time)}: {key} The gap between the end time and the last glucose reading is {end_gap} minutes.")
        #return None, None
    # Subtract the baseline value from all readings
    baseline = post_meal_data[gluc_var].iloc[0]  # Assuming the first value is the baseline
    post_meal_data.loc[:, 'adjusted_val'] = post_meal_data[gluc_var] - baseline
    # Convert 'time' to a series of time intervals in minutes from the closest time
    time_intervals = (post_meal_data['time'] - pd.to_datetime(closest_time)).dt.total_seconds() / 60
    # Calculate the area using the trapezoidal rule for values above the baseline
    

    ######## PPGR metrics
    tc, vc = _split_at_baseline(time_intervals.to_numpy(), post_meal_data['adjusted_val'].to_numpy())
    positive_iAUC = round(np.trapz(np.clip(vc, 0, None), x=tc), 3)
    negative_iAUC = round(np.trapz(np.clip(vc, None, 0), x=tc), 3)

    #positive_iAUC = round(np.trapezoid(post_meal_data['adjusted_val'].clip(lower=0), x=time_intervals), 3)
    #negative_iAUC = round(np.trapezoid(post_meal_data['adjusted_val'].clip(upper=0), x=time_intervals), 3)
    ppgr_array = post_meal_data[gluc_var].to_list()
    net_iAUC = round(np.trapz(post_meal_data["adjusted_val"], x=time_intervals), 3)

    # delta glucose
    max_idx = post_meal_data[gluc_var].idxmax()
    max_glucose = post_meal_data.loc[max_idx, gluc_var]
    delta_glucose = max_glucose - baseline
    # time to peak
    time_of_max = post_meal_data.loc[max_idx, "time"]
    time_to_max = (
        pd.to_datetime(time_of_max) - pd.to_datetime(closest_time)
    ).total_seconds() / 60

    # time above baseline
    peak_duration = calculate_time_above_baseline(
                                    post_meal_data["time"],
                                    post_meal_data[gluc_var],
                                    baseline
                                )

    # recovery fraction: how much of the peak excursion is still present at window end
    # R ~ 0 -> fully recovered to baseline; R ~ 1 -> still near peak (monotonic responses)
    DELTA_FLOOR = 1.0  # mg/dL; below this, response is too flat for R to be meaningful

    # value at the fixed window end (t = post_meal_duration), interpolated onto the grid.
    # adjusted_val is already (glucose - baseline), so this is (G_end - baseline) directly.
    residual_end = float(np.interp(post_meal_duration, time_intervals, post_meal_data['adjusted_val']))
    end_delta = round(residual_end, 3)
    end_glucose = round(baseline + residual_end, 3)


    if delta_glucose >= DELTA_FLOOR:
        recovery_percentage = round(1 - (residual_end / delta_glucose), 3)
    else:
        recovery_percentage = 0  # non-response: R undefined, flag rather than divide


    # 0->60mn burden and 60->120mn burden:
    # split the net iAUC into 0–60 and 60–120 min windows
    split = 60
    # interpolate the value at the 60-min boundary so it's shared by both windows
    val_at_split = float(np.interp(split, time_intervals, post_meal_data['adjusted_val']))

    t = time_intervals.to_numpy()
    v = post_meal_data['adjusted_val'].to_numpy()

    ppgr_time_array = time_intervals.to_list()   # minutes from meal time (t=0), same length/order as ppgr_array


    # include the exact 60-min point as a boundary node in each window
    t0, v0 = np.append(t[t < split], split),       np.append(v[t < split], val_at_split)
    t1, v1 = np.insert(t[t > split], 0, split),     np.insert(v[t > split], 0, val_at_split)

    net_iAUC_0_60   = round(np.trapz(v0, x=t0), 3)
    net_iAUC_60_120 = round(np.trapz(v1, x=t1), 3)

    # nadir: absolute minimum glucose post-peak
    # nadir_delta: nadir minus baseline (positive = stayed above baseline,
    #              negative = dropped below baseline / reactive dip)
    # literature standard: nadir is absolute, delta is relative to baseline

    post_peak_data = post_meal_data[post_meal_data['time'] > time_of_max].copy()

    if len(post_peak_data) == 0:
        # peak at last measurement — no post-peak window
        nadir_glucose = max_glucose          # best we can do
        nadir_delta   = round(max_glucose - baseline, 3)  # still above baseline

    elif delta_glucose < DELTA_FLOOR:
        # no meaningful excursion — nadir is meaningless as a reactive dip metric
        # report the absolute nadir but flag delta as 0
        nadir_glucose = round(post_peak_data[gluc_var].min(), 3)
        nadir_delta   = 0.0

    else:
        # normal case: real excursion occurred
        nadir_glucose = round(post_peak_data[gluc_var].min(), 3)
        nadir_delta   = round(nadir_glucose - baseline, 3)
        # nadir_delta > 0 : recovered but stayed above baseline (no reactive dip)
        # nadir_delta < 0 : dropped below baseline (reactive hypoglycemia)
        # nadir_delta = 0 : recovered exactly to baseline

    # ascending slope
    ASCENT_FLOOR = 1.0  # Minimum excursion for the ratio, mg/dL.

    if delta_glucose < ASCENT_FLOOR:          # subsumes time_to_max == 0
        ascending_slope = 0.0
    else:
        ascending_slope = round(delta_glucose / time_to_max, 4)

    # recovery slope: rate of glucose descent after peak (mg/dL per minute)
    # computed over the post-peak portion of the window
    # negative = recovering

    DELTA_FLOOR_SLOPE = 1.0  # same guard as recovery_percentage

    desc = post_meal_data[post_meal_data['time'] >= time_of_max]
    tt   = (desc['time'] - pd.to_datetime(closest_time)).dt.total_seconds().to_numpy() / 60
    gg   = desc[gluc_var].to_numpy()

    if len(gg) < 2 or delta_glucose < ASCENT_FLOOR:
        recovery_slope = 0.0
    else:
        seg = np.diff(gg) / np.diff(tt)
        recovery_slope = round(min(float(np.min(seg)), 0.0), 4)   # most negative = fastest fall

    return positive_iAUC, baseline, pd.to_datetime(closest_time), ppgr_array, net_iAUC, negative_iAUC, time_to_max, delta_glucose, peak_duration, recovery_percentage, net_iAUC_0_60, net_iAUC_60_120, recovery_slope, nadir_delta, end_delta, max_glucose, ppgr_time_array, end_glucose


## 4. Define dietary-history and glucose-history calculations

Prepare participant-level functions for prior nutrient totals and premeal glucose summaries.

In [11]:
from sklearn.linear_model import LinearRegression

#### @@@@ SHIFT RELATED
def compute_log_glucose(df_food, df_glucose):
    '''
        Compute the log glucose variables as described in Zeevi et al. supp material
            Parameters:
                    df_food (DataFrame): The output data from the function "prepare_dataset_all"
                    df_glucose (DataFrame): The output data from the function "prepare_dataset_all"
            Returns:
                     complete_data (DataFrame): writes the output to a csv file
    '''
    print('computing log glucose variables, please wait...')

    def compute_regression_trend(glu_df, end_time, hours=1):
        start_time = end_time - timedelta(hours=hours)
        window = glu_df[(glu_df['time'] >= start_time) & (glu_df['time'] <= end_time)]
        if len(window) < 2:
            return np.nan  # not enough data
        
        # Convert timestamps to numeric (minutes since start)
        X = (window['time'] - window['time'].min()).dt.total_seconds().values.reshape(-1, 1)
        y = window['mg_dL'].values
        
        reg = LinearRegression().fit(X, y)
        return reg.coef_[0] * 3600  # slope in mg/dL per hour

    complete_data = []
    for identity in tqdm(list(df_food['subject_key'].unique())):
        # subset by key
        my_foods = df_food.loc[df_food['subject_key']==identity]
        my_glu = df_glucose.loc[df_glucose['subject_key']==identity]
        my_foods = my_foods.sort_values(by="eaten_at")
        my_glu = my_glu.sort_values(by="time")

        my_foods['closest_end_time'] = shift_times_to_argrelmin(my_foods, my_glu, timewindow=30, gluc_var="mg_dL")["shifted_eaten_at"].values
        my_foods['glu_end'] =  my_foods['closest_end_time'].apply(lambda x: retrieve_gluc_vals(x, my_glu))
        
        for hour in [1, 2, 4, 6]:
            
            my_foods['start_'+str(hour)] = my_foods['closest_end_time'].copy() - timedelta(hours=hour)

            
            my_foods['glu_start'] =  my_foods['start_'+str(hour)].apply(lambda x: retrieve_gluc_vals(x, my_glu))
            
            my_foods["trend_glu_"+str(hour)] = my_foods['closest_end_time'].apply(
                lambda t: compute_regression_trend(my_glu, t, hours=hour)
            )

            # calculates the iAUC trend
            prior_iauc_baseline_vals = my_foods['start_'+str(hour)].apply(lambda x:
                                                    calculate_PPGR_iAUC(my_glu, identity, x,
                                                    post_meal_duration=60*hour, verbose=False))
                        
            my_foods['iAUC_-'+str(hour)] = [i[0] for i in prior_iauc_baseline_vals]
            

        complete_data.append(my_foods)
    complete_data = pd.concat(complete_data, axis = 0)
    complete_data = complete_data.drop(['glu_start', 'glu_end',
                                        'start_1', 'start_2', 'start_4', ], axis = 1)
    return complete_data

In [12]:
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
from tqdm import tqdm
from datetime import timedelta
import pandas as pd
import numpy as np

def compute_regression_trend(glu_df, end_time, hours=1):
    start_time = end_time - timedelta(hours=hours)
    window = glu_df[
        (glu_df['time'] >= start_time) &
        (glu_df['time'] <= end_time)
    ]
    if len(window) < 2:
        return np.nan
    X = (
        window['time'] - window['time'].min()
    ).dt.total_seconds().values.reshape(-1, 1)

    y = window['mg_dL'].values

    reg = LinearRegression().fit(X, y)

    return reg.coef_[0] * 3600

def compute_mean_gluc(glu_df, end_time, hours=1):
    start_time = end_time - timedelta(hours=hours)
    window = glu_df[
        (glu_df['time'] >= start_time) &
        (glu_df['time'] <= end_time)
    ]
    if len(window) < 2:
        return np.nan
    y = window['mg_dL'].values
    
    y = y[~np.isnan(y)]
    mean = np.mean(y)
    
    return mean

def process_subject_gluc(identity, df_food, df_glucose, timewindow, eating_time_col="eaten_at"):

    # subset by key
    my_foods = df_food[df_food['subject_key'] == identity].copy()
    my_glu = df_glucose[df_glucose['subject_key'] == identity].copy()

    my_foods = my_foods.sort_values(by=eating_time_col)
    my_glu = my_glu.sort_values(by="time")

    my_foods['closest_end_time'] = shift_times_to_argrelmin(
        my_foods,
        my_glu,
        timewindow,
        gluc_var="mg_dL",
        eating_time_col=eating_time_col,
    )["shifted_eaten_at"].values
    my_foods['glu_end'] = my_foods['closest_end_time'].apply(
        lambda x: retrieve_gluc_vals(x, my_glu)
    )
    for hour in [1, 2, 4, 6]:
        my_foods[f'start_{hour}'] = (
            my_foods['closest_end_time'] - timedelta(hours=hour)
        )
        my_foods['glu_start'] = my_foods[f'start_{hour}'].apply(
            lambda x: retrieve_gluc_vals(x, my_glu)
        )
        my_foods[f"trend_glu_{hour}"] = my_foods[
            'closest_end_time'
        ].apply(
            lambda t: compute_regression_trend(
                my_glu,
                t,
                hours=hour
            )
        )
        my_foods[f"mean_glu_{hour}"] = my_foods[
            'closest_end_time'
        ].apply(
            lambda t: compute_mean_gluc(
                my_glu,
                t,
                hours=hour
            )
        )
        prior_iauc_baseline_vals = my_foods[f'start_{hour}'].apply(
            lambda x: calculate_PPGR_iAUC(
                my_glu,
                identity,
                x,
                post_meal_duration=60 * hour,
                verbose=False
            )
        )
        my_foods[f'iAUC_-{hour}'] = [
            i[0] for i in prior_iauc_baseline_vals
        ]
    return my_foods

from tqdm_joblib import tqdm_joblib
timewindow=30
def compute_log_glucose_parallel(df_food, df_glucose, eating_time_col="eaten_at",n_jobs=-1):

    print('computing log glucose variables in parallel...')

    identities = df_food['subject_key'].unique()

    with tqdm_joblib(
        tqdm(desc="Processing subjects", total=len(identities))
    ):
        results = Parallel(n_jobs=n_jobs)(
            delayed(process_subject_gluc)(
                identity,
                df_food,
                df_glucose,
                timewindow=30,
                eating_time_col=eating_time_col,
            )
            for identity in identities
        )

    complete_data = pd.concat(results, axis=0)

    return complete_data

In [13]:
def compute_log_food(df_food, df_gluc, shift_time_extent=30):
    '''
        Compute the log glucose variables as described in Zeevi et al. supp material
            Parameters:
                    df_food (DataFrame): The output data from the function "prepare_dataset_all"
                    df_glucose (DataFrame): The output data from the function "prepare_dataset_all"
            Returns:
                     complete_data (DataFrame): writes the output to a csv file
    '''
    complete_data = []
    for identity in tqdm(list(df_food['subject_key'].unique())):
        my_foods = df_food.loc[df_food['subject_key']==identity]
        my_glu = df_gluc.loc[df_gluc['subject_key']==identity]
        my_foods['shifted_eaten_at'] = shift_times_to_argrelmin(my_foods, my_glu, shift_time_extent)["shifted_eaten_at"].values
        prev_food_user = []
        for e,row in my_foods.iterrows():
            prev_foods = pd.Series(dtype='float64')
            for hour in [1, 2, 4, 6, 12]:
                food_segment = my_foods[(my_foods['shifted_eaten_at'] >= (row['shifted_eaten_at']-timedelta(hours=hour))) &
                                         (my_foods['shifted_eaten_at'] < (row['shifted_eaten_at']))]
                for nutri_var in ['energy_kcal_eaten','carb_eaten','fat_eaten','protein_eaten','fiber_eaten']:
                    if len(food_segment) == 0:
                        prev_foods[f'prev_{hour}hr_'+nutri_var] = 0
                    else:
                        prev_foods[f'prev_{hour}hr_'+nutri_var] = food_segment[nutri_var].sum()
            prev_food_user.append(prev_foods)
        prev_food_user = pd.DataFrame(prev_food_user)
        prev_food_user.index = my_foods.index
        my_foods = pd.concat([my_foods, prev_food_user], axis=1)
        complete_data.append(my_foods)
    complete_data = pd.concat(complete_data, axis = 0)
    return complete_data

In [14]:
from joblib import Parallel, delayed
from tqdm import tqdm
from datetime import timedelta
import pandas as pd


def process_subject_food(identity, df_food, df_gluc, eating_time_col="eaten_at", shift_time_extent=30):

    my_foods = df_food.loc[df_food['subject_key'] == identity].copy()
    my_glu = df_gluc.loc[df_gluc['subject_key'] == identity]

    my_foods['shifted_eaten_at'] = (
        shift_times_to_argrelmin(
            my_foods,
            my_glu,
            shift_time_extent,
            eating_time_col=eating_time_col,
        )["shifted_eaten_at"].values
    )

    prev_food_user = []

    for _, row in my_foods.iterrows():

        prev_foods = {}

        for hour in [1, 2, 3, 6, 12]:

            food_segment = my_foods[
                (my_foods['shifted_eaten_at'] >=
                 (row['shifted_eaten_at'] - timedelta(hours=hour))) &

                (my_foods['shifted_eaten_at'] <
                 row['shifted_eaten_at'])
            ]

            for nutri_var in [
                'energy_kcal_eaten',
                'carb_eaten',
                'fat_eaten',
                'protein_eaten',
                'fiber_eaten'
            ]:

                key = f'prev_{hour}hr_{nutri_var}'

                if len(food_segment) == 0:
                    prev_foods[key] = 0
                else:
                    prev_foods[key] = food_segment[nutri_var].sum()

        prev_food_user.append(prev_foods)

    prev_food_user = pd.DataFrame(prev_food_user)
    prev_food_user.index = my_foods.index

    my_foods = pd.concat([my_foods, prev_food_user], axis=1)

    return my_foods


def compute_log_food_parallel(df_food, df_gluc, eating_time_col="eaten_at",shift_time_extent=30, n_jobs=-1):

    identities = df_food['subject_key'].unique()

    results = Parallel(n_jobs=n_jobs)(
        delayed(process_subject_food)(
            identity,
            df_food,
            df_gluc,
            shift_time_extent=shift_time_extent,
            eating_time_col=eating_time_col,
        )
        for identity in tqdm(identities)
    )

    complete_data = pd.concat(results, axis=0)

    return complete_data

-----

## 5. Calculate dietary-history and glucose-history features

Apply the history functions to each participant in parallel.

In [15]:
print("Merged foods shape : ",merged_foods_users.shape)
merged_foods_users_log = compute_log_food_parallel(merged_foods_users, df_gluc,eating_time_col="eaten_at", n_jobs=-1)
print("After Prev Nutri shape : ",merged_foods_users_log.shape)

Merged foods shape :  (105769, 47)


100%|██████████| 1002/1002 [02:28<00:00,  6.73it/s]


After Prev Nutri shape :  (105769, 73)


In [16]:
merged_foods_users_log = merged_foods_users_log[~merged_foods_users_log["eaten_at"].isna()]
merged_foods_users_log = compute_log_glucose_parallel(merged_foods_users_log, df_gluc, eating_time_col="eaten_at", n_jobs=-1)
print("After Prev Gluc shape : ",merged_foods_users_log.shape)

computing log glucose variables in parallel...


Processing subjects:   0%|          | 0/1002 [00:00<?, ?it/s]

  0%|          | 0/1002 [00:00<?, ?it/s]

After Prev Gluc shape :  (105769, 92)


## 6. Calculate PPGR outcomes and retain response traces

Derive two-hour glucose responses, remove meals without a response trace and fill missing nutrient totals.

In [17]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def _normalize(res):
    """Turn a skip (None, None) into a full-length NaN row so the
    DataFrame assembly never index-errors."""
    if res is None or res[0] is None:
        return [np.nan] * len(RESULT_COLS)
    return list(res)
    
RESULT_COLS = [
    'positive_iAUC', 'premeal_glucose', 'closest_time', 'ppgr_array',
    'net_iAUC', 'negative_iAUC', 'peak_time', 'delta_max_glucose',
    'peak_duration', 'recovery_percentage', 'net_iAUC_0_60',
    'net_iAUC_60_120', 'recovery_slope', 'nadir_delta', 'delta_end_glucose',
    'max_glucose', "ppgr_time_array", "end_glucose"
]

def process_user_ppgr(user):
    eating_time_col="closest_end_time"
    user_merged = (
        merged_foods_users_log[
            merged_foods_users_log['subject_key'] == user
        ]
        .sort_values(by=eating_time_col)
        .copy()
    )

    user_glu = df_gluc[df_gluc['subject_key'] == user]

    results = user_merged['closest_end_time'].apply(
        lambda x: _normalize(
            calculate_PPGR_iAUC_minima(
                user_glu,
                user,
                x,
                gluc_threshold=0.01,
                post_meal_duration=120,
                verbose=False,
                timeshift=0,
                minima=False,
                interpolation=True,
                interp_tol_min=15,
                max_gap_min=45,
            )
        )
    )
    res_df = pd.DataFrame(
        results.tolist(),
        index=user_merged.index,
        columns=RESULT_COLS
    )

    return pd.concat([user_merged, res_df], axis=1)

users = merged_foods_users_log['subject_key'].unique()

results = Parallel(
    n_jobs=-1,      # all cores
    backend="loky"
)(
    delayed(process_user_ppgr)(user)
    for user in tqdm(users)
)

users_merged_log_iauc = pd.concat(results, axis=0)

  0%|          | 0/1002 [00:00<?, ?it/s]

Processing subjects:   0%|          | 0/1002 [04:29<?, ?it/s]


In [18]:
import re
users_merged_log_iauc["food_id_list"] = (

    users_merged_log_iauc["food_id"]

    .astype(str)

    .apply(lambda s: list(map(int, re.findall(r"\d+", s))))
)

users_merged_log_iauc[users_merged_log_iauc["food_id_list"].apply(
    lambda x: len(x) > 0 and set(x).issubset({3080, 3240})
)][["combined_name", "food_id"]].value_counts()

combined_name                                   food_id     
['Glucose drink 50g', 'Tag standardised meal']  [3080, 3240]    1013
['Tag standardised meal', 'Glucose drink 50g']  [3240, 3080]     989
['Glucose drink 50g']                           [3080]            18
Name: count, dtype: int64

In [19]:
# remove meals without PPGR trace
users_merged_log_iauc = users_merged_log_iauc[
    ~users_merged_log_iauc["ppgr_array"].isna()].reset_index(drop=True)

In [20]:
summable_features = ['eaten_quantity_in_gram','water', 'dairy_products_meat_fish_eggs_tofu', 
                     'vegetables_fruits','sweets_salty_snacks_alcohol', 
                     'non_alcoholic_beverages', 'grains_potatoes_pulses', 'oils_fats_nuts', #"composite-foods"
                    ]
summable_features += [i for i in users_merged_log_iauc.columns 
                      if "_eaten" in i if i not in ["local_eaten_at", 'energy_kj_eaten']]

users_merged_log_iauc[summable_features] = users_merged_log_iauc[summable_features].fillna(0)

users_merged_log_iauc.shape

(99059, 111)

## 7. Calculate meal intervals and clock-time features

Calculate time since the previous meal, time until the next meal, and time-of-day and weekday variables.

In [21]:
users_merged_log_iauc["time_diff"] = (
    pd.to_datetime(users_merged_log_iauc["closest_end_time"]) 
    - pd.to_datetime(users_merged_log_iauc["shifted_eaten_at"])
)

#users_merged_log_iauc["time_diff"].describe()

In [22]:
# Sort meals chronologically before calculating intervals.
users_merged_log_iauc["shifted_eaten_at"] = pd.to_datetime(users_merged_log_iauc["shifted_eaten_at"])
users_merged_log_iauc = users_merged_log_iauc.sort_values(['subject_key', 'shifted_eaten_at'])

g = users_merged_log_iauc.groupby('subject_key')['shifted_eaten_at']

# Interval since the previous meal, in hours.
users_merged_log_iauc['time_since_last_meal'] = g.diff().dt.total_seconds() / 3600

users_merged_log_iauc['time_since_last_meal'] = users_merged_log_iauc['time_since_last_meal'].where(
    users_merged_log_iauc['time_since_last_meal'] <= 24, np.nan)

In [23]:
# Interval until the next meal, in hours.
# gap to next meal at row i == backward gap at row i+1, within the same subject.
users_merged_log_iauc['time_to_next_meal'] = (
    users_merged_log_iauc
    .groupby('subject_key')['time_since_last_meal']   # groupby shift respects boundaries
    .shift(-1)
)

In [24]:
users_merged_log_iauc[(users_merged_log_iauc["subject_key"]=="02ae3856ca04")][
    ["subject_key", "closest_end_time",
     "time_to_next_meal","time_since_last_meal"]
].head(3)

,subject_key,closest_end_time,time_to_next_meal,time_since_last_meal
0,02ae3856ca04,2018-11-26 08:56:24,1.683889,NaN
1,02ae3856ca04,2018-11-26 10:37:26,1.078611,1.683889
2,02ae3856ca04,2018-11-26 11:42:09,2.547500,1.078611


In [25]:
users_merged_log_iauc["closest_end_time"] = pd.to_datetime(users_merged_log_iauc["closest_end_time"])
users_merged_log_iauc["minutes_of_the_day"] = users_merged_log_iauc["closest_end_time"].dt.hour * 60 + users_merged_log_iauc["closest_end_time"].dt.minute
users_merged_log_iauc["hours_numeric"] = users_merged_log_iauc["closest_end_time"].dt.hour + users_merged_log_iauc["closest_end_time"].dt.minute /60
users_merged_log_iauc['weekday_index'] = users_merged_log_iauc['closest_end_time'].dt.weekday 

## 8. Identify standardized meals

Classify the retained glucose-drink, white-bread and white-bread-with-butter challenges using food identifiers and carbohydrate amounts.

In [26]:
import re

allowed_ids = {3240, 3080, 1566, 2053}

# Extract IDs
users_merged_log_iauc["food_id_list"] = (
    users_merged_log_iauc["food_id"]
    .astype(str)
    .apply(lambda s: list(map(int, re.findall(r"\d+", s))))
)

print("ok")

# Flag standardized meals
users_merged_log_iauc["is_standardized_meal"] = (
    users_merged_log_iauc["food_id_list"]
    .apply(lambda x: len(x) > 0 and set(x).issubset(allowed_ids))
    .astype(int)
)

# Assign meal type
users_merged_log_iauc["standardized_meal_id"] = users_merged_log_iauc["food_id_list"].apply(
    lambda x:
        "other" if not (len(x) > 0 and set(x).issubset(allowed_ids)) else
        "other" if ({3080, 1566} <= set(x)) else
        "C"     if ({1566, 2053} <= set(x)) else
        "B"     if 1566 in x else
        "A"     if 3080 in x else
        "other"
)

users_merged_log_iauc["standardized_meal_id"].value_counts()

ok


standardized_meal_id
other    93779
A         1973
B         1795
C         1512
Name: count, dtype: int64

In [27]:
# process standardized meals

import re

allowed_ids = {3240, 3080, 1566, 2053}

# Extract IDs
users_merged_log_iauc["food_id_list"] = (
    users_merged_log_iauc["food_id"]
    .astype(str)
    .apply(lambda s: list(map(int, re.findall(r"\d+", s))))
)

# Assign meal type
def assign_standardized_meal(row):
    ids = set(row["food_id_list"])
    carb = row["carb_eaten"]

    # A: 3080 or (3080 + 3240), carb between 45 and 55
    if ids in ({3080}, {3080, 3240}) and 45 <= carb <= 55:
        return "A"

    # B: 1566 or (1566 + 3240), carb between 40 and 60
    if ids in ({1566}, {1566, 3240}) and 45 <= carb <= 55:
        return "B"

    # C: 2053 or (2053 + 3240), carb between 40 and 60
    if ids in ({1566, 2053}, {1566, 2053, 3240}) and 45 <= carb <= 55:
        return "C"

    return "other"


users_merged_log_iauc["standardized_meal_id"] = users_merged_log_iauc.apply(
    assign_standardized_meal,
    axis=1
)

# Flag standardized meals
users_merged_log_iauc["is_standardized_meal"] = (
    users_merged_log_iauc["standardized_meal_id"]
    .ne("other")
    .astype(int)
)
users_merged_log_iauc["standardized_meal_id"].value_counts()

standardized_meal_id
other    94107
A         1888
B         1628
C         1436
Name: count, dtype: int64

In [28]:
users_merged_log_iauc.shape

(99059, 119)

## 9. Export the prepared meal table

Write the table to the CSV or ZIP path selected by `MEAL_DATA_PATH`.

In [29]:
users_merged_log_iauc.to_csv(MEAL_DATA_PATH)